# 08 | Publicação e geração operacional

POC semissintética • contrato provisório • sem homologação acadêmica.
Execute em kernel reiniciado, de cima para baixo. As credenciais vêm do ambiente.


**Objetivo:** disponibilizar versões aprovadas e gerar a proposta pela API. **Entradas:** campanha e decisões07. Por padrão consulta apenas; altere os controles explicitamente. **Esperado:** versão publicada distinta de residente.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.application_client import call, context, save_context

from src.workbench.academic_ui import campaign_result
result=campaign_result()
DEPLOY_DEMO=os.getenv('V4_DEPLOY_DEMO','false').lower()=='true'
GENERATE=os.getenv('V4_GENERATE','false').lower()=='true'
GENERATION_KEY=os.getenv('V4_GENERATION_KEY','v4-proposal-'+result['campaign_id'])
selectors=[{k:m[k] for k in ('model_name','model_version')} for m in result['models']]
if DEPLOY_DEMO:
    for index,selector in enumerate(selectors):
        call('POST',f"/v1/serving/models/{selector['model_name']}/deploy",dict(**selector,policy='PRELOAD' if index==0 else 'LAZY'),
             key=f"v4-notebook-deploy-{selector['model_name']}-{selector['model_version']}",operator=True)
display(pd.DataFrame(call('GET','/v1/serving/models')))
display(call('GET','/v1/serving/runtime'))


,model_key,model_version,course_code,validation_status,run_id,snapshot_id,approval_status,deployment_status,policy,expires_at
0,hi.demo.v4.c2248.components,1,2248,VALIDATED,4f0921ac71de407f98427e1e5fe22f3c,9158d4db-c80b-5638-8aa0-9467c60f6107,None,None,None,None
1,hi.demo.v4.c2248.components,2,2248,VALIDATED,3bdb8b358a3c45e2b0491bafebe134fe,9158d4db-c80b-5638-8aa0-9467c60f6107,None,None,None,None
2,hi.demo.v4.c2248.components,3,2248,VALIDATED,5eee1810f09c40e088b1ed061d517ea1,9158d4db-c80b-5638-8aa0-9467c60f6107,APPROVED,DEPLOYED,PRELOAD,2026-12-05T07:24:39.881623+00:00
3,hi.demo.v4.c2287.components,1,2287,VALIDATED,b6add4f963cd42579b92da413349ae11,9158d4db-c80b-5638-8aa0-9467c60f6107,APPROVED,DEPLOYED,LAZY,2026-12-05T07:24:40.300473+00:00
4,hi.demo.v4.c2348.components,1,2348,VALIDATED,18e7787eed2043f98756ba71ee58ff7e,9158d4db-c80b-5638-8aa0-9467c60f6107,APPROVED,DEPLOYED,LAZY,2026-12-05T07:24:40.095920+00:00


{'versions': [],
 'cache': {'hits': 0, 'misses': 0, 'errors': 0, 'currsize': 0, 'maxsize': 4}}

**Executar:** geração HTTP, aguardando job persistido. **Resultado:**70 ofertas/107 origens, `proposal_id` salvo automaticamente para09. Mesma chave recupera; nova chave gera nova proposta, sem sobrescrever anteriores.

In [2]:

import time
if GENERATE:
    payload=dict(schema_version='poc-planilha-turmas-v2',snapshot_id=result['snapshot_id'],cycle_code='2024-2',proposal_revision=1,
                 model_selectors=selectors,requested_by='jupyter-poc')
    job=call('POST','/v2/proposals/generations',payload,key=GENERATION_KEY)
    for _ in range(90):
        job=call('GET',job['status_url'])
        if job['status'] in ('COMPLETED','FAILED'):
            break
        time.sleep(2)
    if job['status']!='COMPLETED':
        raise RuntimeError('Geração não concluída: '+str({k:job[k] for k in ('job_id','status','error_code')}))
    save_context(proposal_id=job['proposal_id'])
    display({k:job[k] for k in ('job_id','proposal_id','status','proposal_url')})
else:
    display('READ_ONLY: geração não solicitada. Use os controles após aprovação.')


{'job_id': '1be86675-ad44-4436-9e7f-a6c49bfc5b81',
 'proposal_id': '1be86675-ad44-4436-9e7f-a6c49bfc5b81',
 'status': 'COMPLETED',
 'proposal_url': '/v2/proposals/1be86675-ad44-4436-9e7f-a6c49bfc5b81/application-view'}

**Onde conferir:** [MLflow](http://localhost:5000), [RustFS](http://localhost:9001), [API](http://localhost:8000/docs), [Logs](http://localhost:8088). Em falha: `docker compose logs --tail 100 api inference worker mlflow`.